# Importación de Librerias

Para la API de steam es necesario obtenerla, para ello se debe contar con una cuenta en Steam.

El link para obtener la API es la sigueinte: [obtén una aquí](https://steamcommunity.com/dev/apikey)

Omitir si no es necesario obtener los datos de la API. y pasar al 02, ya que la data se subio a Drive

### Descarga de librerias

### Importamos las librerias necesarias

In [1]:
# For Production
import json
import random
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Librerias necesarias para funciones
from pathlib import Path
from howlongtobeatpy import HowLongToBeat

# Instalar primero: pip install python-dotenv
from dotenv import load_dotenv
import os

import time
from datetime import datetime # unlooktime

In [2]:
# Configuración de entorno
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
# Crear carpeta data si no existe
os.makedirs('data', exist_ok=True)

# Definicion de variables para API

In [4]:
STEAM_API_KEY = os.getenv('STEAM_API_KEY')
STEAM_ID = os.getenv('STEAM_ID')

# Definición de Funciones

Definimos las funciones ocupadas

### Obtener id's de Juegos RPG

In [5]:
def get_all_games_by_genre(genre: str, max_retries: int = 3):
    """
    Obtiene TODOS los juegos de SteamSpy para un género específico en UNA sola llamada.

    Parámetros:
    - genre: Género a buscar (RPG, Action, Adventure, etc.)
    - max_retries: Número máximo de reintentos en caso de error

    Retorna: diccionario con todos los juegos {appid: {info}}

    Nota: SteamSpy devuelve TODOS los juegos del género en una sola respuesta (no hay paginación).
    """
    from requests.exceptions import ConnectionError, Timeout, RequestException

    print(f"🎮 Descargando todos los juegos del género: {genre}")

    url = "https://steamspy.com/api.php"
    params = {"request": "genre", "genre": genre}

    # Intentar hasta max_retries veces
    for intento in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()

            if not data:
                print("⚠️ La respuesta está vacía")
                return {}

            # Estructurar los datos
            juegos = {}
            for appid, info in data.items():
                juegos[appid] = {
                    # Identificación
                    "appid": appid,
                    "nombre": info.get("name"),

                    # Desarrollador y publisher
                    "developer": info.get("developer"),
                    "publisher": info.get("publisher"),

                    # Ranking y propietarios
                    "score_rank": info.get("score_rank"),
                    "owners": info.get("owners"),

                    # Tiempo de juego (en minutos)
                    "average_forever": info.get("average_forever"),
                    "average_2weeks": info.get("average_2weeks"),
                    "median_forever": info.get("median_forever"),
                    "median_2weeks": info.get("median_2weeks"),

                    # Usuarios concurrentes
                    "ccu": info.get("ccu"),

                    # Precios (en centavos USD)
                    "price": info.get("price"),
                    "initialprice": info.get("initialprice"),
                    "discount": info.get("discount"),

                    # Tags, idiomas y géneros
                    "tags": info.get("tags"),
                    "languages": info.get("languages"),
                    "genre": info.get("genre"),

                    # Género de búsqueda (para filtrado posterior)
                    "genero_busqueda": genre,
                }

            print(f"✅ Descarga completada: {len(juegos)} juegos del género {genre}")
            return juegos

        except (ConnectionError, Timeout) as e:
            error_tipo = "DNS/Conexión" if "Failed to resolve" in str(e) else "Timeout"
            if intento < max_retries - 1:
                espera = 2 ** intento  # backoff exponencial: 1s, 2s, 4s
                print(f"⚠️ Error {error_tipo}, reintento {intento+1}/{max_retries} en {espera}s...")
                time.sleep(espera)
            else:
                print(f"❌ Error {error_tipo} después de {max_retries} intentos: {e}")
                if "Failed to resolve" in str(e):
                    print("   💡 Solución: Verifica tu conexión a Internet o DNS")
                return {}

        except RequestException as e:
            print(f"❌ Error HTTP: {e}")
            return {}

        except Exception as e:
            print(f"❌ Error inesperado: {e}")
            return {}

    return {}

### Tiempo promedio en Terminar un Juego

In [6]:
def get_game_duration(game_name: str):
    """
    Obtiene tiempos de juego desde HowLongToBeat.

    Retorna:
    - main_story: Tiempo historia principal (horas)
    - main_extra: Historia + extras (horas)
    - completionist: 100% completado (horas)
    """
    results = HowLongToBeat().search(game_name)

    if results:
        game = results[0]  # Primer resultado
        return {
            'name': game.game_name,
            'main_story': game.main_story,
            'main_extra': game.main_extra,
            'completionist': game.completionist
        }
    return None

### Jugadores (reviews) - Final

In [7]:
def get_reviews_batch_for_game(appid: int, cursor: str = '*',
                               batch_size: int = 1000,
                               review_type: str = 'all',
                               language: str = 'latam',
                               rate_limit_seconds: float = 1.0):
    """
    Obtiene un batch (lote) de reseñas de un juego.

    Parámetros:
    - appid: ID del juego en Steam
    - cursor: Cursor de paginación (usar '*' para el primer batch)
    - batch_size: Número de reseñas a obtener en este batch
    - review_type: 'positive', 'negative', 'all'
    - language: Código de idioma ('latam' para español latinoamericano)
    - rate_limit_seconds: Segundos de espera entre requests

    Retorna:
    - Tupla (lista_reseñas, next_cursor, query_summary)
    """
    url = f"https://store.steampowered.com/appreviews/{appid}"
    reviews_collected = []
    current_cursor = cursor
    query_summary = None

    # Obtener reseñas en bloques de 100 (máximo por request) hasta alcanzar batch_size
    while len(reviews_collected) < batch_size:
        params = {
            'json': 1,
            'filter': review_type,
            'language': language,
            'purchase_type': 'all',
            'num_per_page': 100,  # Máximo por request
            'cursor': current_cursor
        }

        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()

            if not data.get('success'):
                break

            # Capturar query_summary del primer request
            if query_summary is None and 'query_summary' in data:
                query_summary = data['query_summary']

            reviews = data.get('reviews', [])
            if not reviews:
                # No hay más reseñas
                break

            reviews_collected.extend(reviews)

            # Obtener siguiente cursor
            current_cursor = data.get('cursor')
            if not current_cursor:
                break

            # Si ya alcanzamos el batch_size, salir
            if len(reviews_collected) >= batch_size:
                break

            # Rate limiting
            time.sleep(rate_limit_seconds)

        except Exception as e:
            print(f"❌ Error obteniendo batch de appid {appid}: {e}")
            break

    # Limitar al batch_size exacto
    if len(reviews_collected) > batch_size:
        reviews_collected = reviews_collected[:batch_size]

    return reviews_collected, current_cursor, query_summary

In [8]:
def get_reviews_for_multiple_games_progressive(appids: list,
                                               checkpoint_file: str = "data/reviews_checkpoint.parquet",
                                               batch_size: int = 1000,
                                               max_reviews_per_game: int = None,
                                               max_games: int = None,
                                               checkpoint_every_round: int = 1,
                                               rate_limit_seconds: float = 1.0,
                                               language: str = 'latam'):
    """
    Obtiene SOLO author.steamid de reseñas de múltiples juegos (progresivo).

    Versión simplificada que solo extrae el Steam ID de cada autor.
    Guarda checkpoints en formato Parquet.

    Retorna: DataFrame con columnas [game_appid, author_steamid]
    """

    df_steamids = pd.DataFrame(columns=['game_appid', 'author_steamid'])
    game_cursors = {}
    game_review_counts = {}
    completed_games = set()
    round_number = 0

    # Archivos de checkpoint
    parquet_file = checkpoint_file
    meta_file = checkpoint_file.replace('.parquet', '_meta.json')

    # Cargar checkpoint
    if Path(parquet_file).exists():
        df_steamids = pd.read_parquet(parquet_file)
        print(f"📂 Datos cargados: {len(df_steamids):,} registros")

    if Path(meta_file).exists():
        with open(meta_file, 'r') as f:
            meta = json.load(f)
            game_cursors = meta.get('cursors', {})
            game_review_counts = meta.get('counts', {})
            completed_games = set(meta.get('completed', []))
            round_number = meta.get('round', 0)
            print(f"📂 Reanudando ronda {round_number + 1}")

    if max_games:
        appids = appids[:max_games]

    # Inicializar
    for appid in appids:
        if str(appid) not in game_cursors:
            game_cursors[str(appid)] = '*'
            game_review_counts[str(appid)] = 0

    active_games = [a for a in appids if a not in completed_games]
    print(f"\n🎮 {len(appids):,} juegos | Batch: {batch_size:,}\n")
    inicio = time.time()

    # Procesar por rondas
    while active_games:
        round_number += 1
        print(f"🔄 RONDA {round_number} - {len(active_games)} activos")
        newly_completed = []

        for idx, appid in enumerate(active_games, 1):
            appid_str = str(appid)
            cursor = game_cursors.get(appid_str, '*')
            count = game_review_counts.get(appid_str, 0)

            # Verificar límite
            if max_reviews_per_game and count >= max_reviews_per_game:
                newly_completed.append(appid)
                continue

            reviews_to_get = batch_size
            if max_reviews_per_game:
                reviews_to_get = min(batch_size, max_reviews_per_game - count)

            print(f"[{idx}/{len(active_games)}] AppID {appid}...", end=' ')

            # Obtener batch
            reviews, next_cursor, _ = get_reviews_batch_for_game(
                appid=appid,
                cursor=cursor,
                batch_size=reviews_to_get,
                language=language,
                rate_limit_seconds=rate_limit_seconds
            )

            if reviews:
                # Extraer SOLO author.steamid
                new_rows = []
                for review in reviews:
                    steamid = review.get('author', {}).get('steamid')
                    if steamid:
                        new_rows.append({
                            'game_appid': appid,
                            'author_steamid': steamid
                        })

                # Agregar nuevos registros al DataFrame
                if new_rows:
                    df_new = pd.DataFrame(new_rows)
                    df_steamids = pd.concat([df_steamids, df_new], ignore_index=True)

                game_review_counts[appid_str] += len(reviews)
                game_cursors[appid_str] = next_cursor
                print(f"+{len(reviews)} (total: {game_review_counts[appid_str]:,})")
            else:
                print(f"Completado ({count:,})")
                newly_completed.append(appid)

        # Actualizar completados
        for appid in newly_completed:
            completed_games.add(appid)
            active_games.remove(appid)

        # Checkpoint en Parquet + JSON (metadatos)
        if round_number % checkpoint_every_round == 0 or not active_games:
            # Guardar datos en Parquet
            df_steamids.to_parquet(parquet_file, index=False, compression='snappy')

            # Guardar metadatos en JSON
            with open(meta_file, 'w') as f:
                json.dump({
                    'cursors': game_cursors,
                    'counts': game_review_counts,
                    'completed': list(completed_games),
                    'round': round_number
                }, f)

            print(f"💾 Checkpoint: {len(df_steamids):,} registros | {time.time()-inicio:.0f}s\n")

    print(f"✅ Completado: {len(df_steamids):,} registros en {(time.time()-inicio)/60:.1f} min")
    print(f"   Steam IDs únicos: {df_steamids['author_steamid'].nunique():,}")
    return df_steamids

### Amigos de Jugadores

In [9]:
def get_friends_list(steam_id, api_key):
    """
    Obtiene la lista de amigos de un Steam ID

    Args:
        steam_id: Steam ID del usuario
        api_key: Clave de API de Steam

    Returns:
        Lista de diccionarios con información de amigos, o [] si perfil privado/sin amigos
    """
    url = f"https://api.steampowered.com/ISteamUser/GetFriendList/v0001/"
    params = {
        'key': api_key,
        'steamid': steam_id,
        'relationship': 'friend'
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        # Cualquier status 2xx se considera éxito
        if 200 <= response.status_code < 300:
            try:
                data = response.json()
                # Si el JSON está vacío o no tiene la estructura esperada
                # significa que el perfil es privado o no tiene amigos
                if not data or 'friendslist' not in data:
                    return []
                if 'friends' in data['friendslist']:
                    return data['friendslist']['friends']
                else:
                    return []
            except Exception as e:
                # Si hay error al parsear el JSON, tratar como sin amigos
                return []
        elif response.status_code == 401:
            print(f"❌ Error 401: API key inválida - verifica STEAM_API_KEY")
            return None
        elif response.status_code == 403:
            # Perfil privado o sin amigos
            return []
        else:
            # Otros errores no críticos
            return []

    except requests.exceptions.RequestException as e:
        print(f"❌ Error de conexión para {steam_id}: {e}")
        return None

In [10]:
def process_steam_ids_with_checkpoint(steam_ids_list, api_key, checkpoint_file='checkpoint_amigos.parquet',
                                     checkpoint_interval=10, delay=1.5):
    """
    Procesa una lista de Steam IDs obteniendo sus amigos, con sistema de checkpoint

    Args:
        steam_ids_list: Lista de Steam IDs a procesar
        api_key: Clave de API de Steam
        checkpoint_file: Nombre del archivo de checkpoint (parquet)
        checkpoint_interval: Cada cuántos IDs guardar checkpoint
        delay: Segundos de espera entre requests (para no saturar la API)

    Returns:
        DataFrame con todos los resultados
    """
    # Intentar cargar checkpoint existente
    if os.path.exists(checkpoint_file):
        print(f"📂 Cargando checkpoint existente: {checkpoint_file}")
        df_checkpoint = pd.read_parquet(checkpoint_file)
        processed_ids = set(df_checkpoint['steam_id'].unique())
        print(f"✓ {len(processed_ids)} Steam IDs ya procesados")
    else:
        df_checkpoint = pd.DataFrame()
        processed_ids = set()
        print("🆕 Iniciando proceso desde cero")

    # Filtrar IDs ya procesados
    pending_ids = [sid for sid in steam_ids_list if sid not in processed_ids]
    print(f"📊 Total a procesar: {len(pending_ids)} Steam IDs")

    if not pending_ids:
        print("✓ Todos los Steam IDs ya fueron procesados")
        return df_checkpoint

    # Lista para almacenar resultados nuevos
    all_results = []

    # Procesar cada Steam ID
    error_count = 0
    for idx, steam_id in enumerate(pending_ids, 1):
        print(f"[{idx}/{len(pending_ids)}] Procesando Steam ID: {steam_id}...", end=' ')

        friends = get_friends_list(steam_id, api_key)

        if friends is not None:
            # Reset contador de errores si funciona
            error_count = 0

            if friends:
                # Agregar el steam_id del usuario a cada registro de amigo
                for friend in friends:
                    friend['steam_id'] = steam_id
                all_results.extend(friends)
                print(f"✓ {len(friends)} amigos encontrados")
            else:
                print("✓ Sin amigos o perfil privado")
                # Guardar registro vacío para marcar como procesado
                all_results.append({
                    'steam_id': steam_id,
                    'steamid': None,
                    'relationship': None,
                    'friend_since': None
                })
        else:
            # Error (401 o error de conexión) - marcar y continuar
            error_count += 1
            print(f"⚠️  Error (total errores: {error_count}), continuando...")
            # Guardar registro de error para no reprocesar
            all_results.append({
                'steam_id': steam_id,
                'steamid': None,
                'relationship': 'error',
                'friend_since': None
            })

        # Guardar checkpoint cada N registros
        if idx % checkpoint_interval == 0:
            if all_results:
                df_new = pd.DataFrame(all_results)
                df_combined = pd.concat([df_checkpoint, df_new], ignore_index=True)
                df_combined.to_parquet(checkpoint_file, index=False)
                print(f"💾 Checkpoint guardado ({len(df_combined)} registros totales)")
                df_checkpoint = df_combined
                all_results = []

        # Delay para no saturar la API de Steam
        time.sleep(delay)

    # Guardar resultados finales
    if all_results:
        df_new = pd.DataFrame(all_results)
        df_combined = pd.concat([df_checkpoint, df_new], ignore_index=True)
    else:
        df_combined = df_checkpoint

    if len(df_combined) > 0:
        df_combined.to_parquet(checkpoint_file, index=False)
        print(f"\n✅ Proceso completado. Total de registros: {len(df_combined)}")
        print(f"💾 Guardado en: {checkpoint_file}")

    return df_combined

### Tiempos Jugados por cada Jugador

In [11]:
def get_owned_games(steam_id: str, api_key: str):
    """
    Devuelve la lista de juegos que posee el usuario con horas jugadas.
    """
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": api_key,
        "steamid": steam_id,
        "include_appinfo": True,       # incluye nombre e ícono del juego
        "include_played_free_games": True,
        "format": "json"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if "response" not in data or "games" not in data["response"]:
        print("Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:")
        print("https://steamcommunity.com/my/edit/settings")
        return []

    games = data["response"]["games"]
    resultado = []
    for g in games:
        resultado.append({
            "appid": g.get("appid"),
            "nombre": g.get("name"),
            "horas_totales": round(g.get("playtime_forever", 0) / 60, 1),  # viene en minutos
            "horas_2_semanas": round(g.get("playtime_2weeks", 0) / 60, 1) if "playtime_2weeks" in g else 0,
        })
    return resultado

In [12]:
def obtener_juegos_todos_jugadores_parquet(df_jugadores, api_key, checkpoint_file='data/checkpoint_juegos_jugadores.parquet',
                                           max_jugadores=None, checkpoint_cada=10, debug=True):
    """
    CON CHECKPOINT: Guarda progreso cada N jugadores y puede reanudar desde donde quedó.
    """
    from pathlib import Path

    jugadores = [int(x) for x in df_jugadores['author_steamid'].dropna().unique()[:max_jugadores]]

    # Cargar checkpoint si existe
    if Path(checkpoint_file).exists():
        print("📂 Cargando checkpoint...")
        df_checkpoint = pd.read_parquet(checkpoint_file)
        procesados = set(df_checkpoint['steam_id'].unique())
        print(f"   ✓ {len(procesados):,} jugadores ya procesados")
    else:
        df_checkpoint = pd.DataFrame()
        procesados = set()

    # Filtrar pendientes (asegurar tipos compatibles)
    pendientes = [j for j in jugadores if int(j) not in procesados]
    total = len(jugadores)
    # Filtrar pendientes (asegurar tipos compatibles)
    pendientes = [j for j in jugadores if int(j) not in procesados]
    total = len(jugadores)

    print(f"🎮 Jugadores: {total:,} | Ya procesados: {len(procesados):,} | Pendientes: {len(pendientes):,}\n")

    if not pendientes:
        print("✅ Todos procesados")
        return df_checkpoint

    nuevos = []
    for idx, steam_id in enumerate(pendientes, 1):
        try:
            juegos = get_owned_games(str(steam_id), api_key)

            if juegos:
                for j in juegos:
                    j['steam_id'] = int(steam_id)
                nuevos.extend(juegos)
                if debug:
                    print(f"✅ [{len(procesados)+idx}/{total}] {steam_id}: {len(juegos)} juegos")
            else:
                nuevos.append({'appid': None, 'nombre': None, 'horas_totales': 0, 'horas_2_semanas': 0, 'steam_id': int(steam_id)})
                if debug:
                    print(f"🔒 [{len(procesados)+idx}/{total}] {steam_id}: Perfil privado")

            # Guardar checkpoint
            if idx % checkpoint_cada == 0:
                df_nuevos = pd.DataFrame(nuevos)
                df_temp = pd.concat([df_checkpoint, df_nuevos], ignore_index=True) if len(df_checkpoint) > 0 else df_nuevos
                Path(checkpoint_file).parent.mkdir(parents=True, exist_ok=True)
                df_temp.to_parquet(checkpoint_file, index=False)
                print(f"💾 Checkpoint guardado")
                df_checkpoint = df_temp
                nuevos = []

            time.sleep(1.1)

        except Exception as e:
            if debug:
                print(f"❌ [{len(procesados)+idx}/{total}] {steam_id}: {str(e)[:40]}")
            nuevos.append({'appid': None, 'nombre': None, 'horas_totales': 0, 'horas_2_semanas': 0, 'steam_id': int(steam_id)})
            time.sleep(1.1)

    # Guardar final
    if nuevos:
        df_nuevos = pd.DataFrame(nuevos)
        df_final = pd.concat([df_checkpoint, df_nuevos], ignore_index=True) if len(df_checkpoint) > 0 else df_nuevos
    else:
        df_final = df_checkpoint

    Path(checkpoint_file).parent.mkdir(parents=True, exist_ok=True)
    df_final.to_parquet(checkpoint_file, index=False)
    print(f"\n✅ Completado: {len(df_final):,} registros | {checkpoint_file}")

    return df_final

### Perfil del jugador

In [13]:
def get_player_summary(steam_id: str, api_key: str):
    """
    Info general del perfil: si es público, última conexión, avatar, etc.
    """
    url = "https://api.steampowered.com/ISteamUser/GetPlayerSummaries/v0002/"
    params = {"key": api_key, "steamids": steam_id, "format": "json"}
    response = requests.get(url, params=params)
    response.raise_for_status()
    players = response.json().get("response", {}).get("players", [])
    return players[0] if players else None

# Juegos RPG

Obtenemos nuestro segmento de juegos en los que nos vamos a efocar para este proyecto.

In [ ]:
juegos_rpg_completos = get_all_games_by_genre(genre="RPG")

En la ultima ejecucion esta aprte salio vacia, tema de la api.

In [32]:
df_rpg_games = pd.DataFrame.from_dict(juegos_rpg_completos, orient='index')

In [ ]:
df_rpg_games['score_rank'] = pd.to_numeric(df_rpg_games['score_rank'], errors='coerce')
df_rpg_games.to_parquet('data/rpg_games.parquet', index=False)

# Tiempo promedio que toma terminar el Juego

Obtenemos el tiempo promedio que toma terminar el Juego, de una api.

In [ ]:
# Nombres únicos (evitar búsquedas repetidas)
nombres_unicos = (
    df_rpg_games["nombre"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .unique()
)

In [ ]:
print(f"🎮 Procesando {len(nombres_unicos):,} juegos únicos...")

🎮 Procesando 15,742 juegos únicos...


In [ ]:
# Aproximadamente 700 min.

# Obtener duración para todos los juegos de df_rpg_games usando get_game_duration()
print(f"🎮 Procesando {len(nombres_unicos):,} juegos únicos...\n")

duraciones_por_nombre = {}

for i, nombre in enumerate(nombres_unicos, start=1):
    # Usar la función get_game_duration con nombre en minúsculas
    duracion = get_game_duration(nombre.lower())

    if duracion:
        duraciones_por_nombre[nombre] = {
            "hltb_name": duracion['name'],
            "main_story": duracion['main_story'],
            "main_extra": duracion['main_extra'],
            "completionist": duracion['completionist'],
        }
    else:
        duraciones_por_nombre[nombre] = {
            "hltb_name": None,
            "main_story": None,
            "main_extra": None,
            "completionist": None,
        }

    if i % 100 == 0:
        print(f"✓ Procesados {i:,}/{len(nombres_unicos):,} nombres")

    time.sleep(0.15)  # evita saturar requests

print(f"\n✅ Completado: {len(nombres_unicos):,} juegos procesados")

🎮 Procesando 10 juegos únicos...


✅ Completado: 10 juegos procesados


In [ ]:
# Convertir diccionario a DataFrame
df_duraciones_nombres = (
    pd.DataFrame.from_dict(duraciones_por_nombre, orient="index")
    .reset_index()
    .rename(columns={"index": "nombre"})
)

In [ ]:
# Unir con df_rpg para tener todas las filas originales + duración
df_rpg_duraciones = df_rpg_games.merge(df_duraciones_nombres, on="nombre", how="left")

In [ ]:
df_rpg_duraciones.to_parquet(r'./data/rpg_juegos_duraciones.parquet', index=False)

# Jugadores que juegan RPG (reviews)

Obtenemos id's de los jugadores que juegan estos juegos mediante una api de rewievs, actualmente no existe una api de la cual podamos obtener los id's de los jugadores por juego.

Para la obtención final de todos los id's obtenidos en este proceso se realizaron varias iteraciónes, las cuales se enlistan a continuación:

> Se ejecuto get_reviews_for_multiple_games_progressive, para obtener un registro de 112K reseñas, de las cuales obtuvimos solo 6K jugadores. La funcion obtenia 3 reseñas de cada juego por cada iteración, solo reseñas de latam. Con esta primera iteración notamos que solo el 30% (4.6K) de los juegos tienen almenos una reseña (api no oficial de steam). Se modifico el guardado de json a parquet, ya que con una cantidad grande de datos "json" no lo leía correctamente, adicionalmente solo obtenemos el "game_appid" y "author_steamid".



In [ ]:
# Primera ejecución - rpg_steamids_prueba
app_reviews_prueba = df_rpg_games['appid'].unique().tolist()
# Se ejecutaron correctamente solo 9 rondas
df_steamids_prueba = get_reviews_for_multiple_games_progressive(
    appids=app_reviews_prueba,
    checkpoint_file="data/rpg_steamids_prueba.parquet",
    batch_size=1,  # 3 reseñas por juego en cada ronda
    max_reviews_per_game=27,  # 27 reseñas (Todas las reseñas es con None)
    checkpoint_every_round=1,  # Guardar checkpoint CADA ronda
    rate_limit_seconds=1.0
    language=None # Todos los lenguajes
)


🎮 10 juegos | Batch: 1

🔄 RONDA 1 - 10 activos
[1/10] AppID 1623730... +1 (total: 1)
[2/10] AppID 1063730... +1 (total: 1)
[3/10] AppID 2358720... +1 (total: 1)
[4/10] AppID 1599340... +1 (total: 1)
[5/10] AppID 2246340... +1 (total: 1)
[6/10] AppID 230410... +1 (total: 1)
[7/10] AppID 1245620... +1 (total: 1)
[8/10] AppID 105600... +1 (total: 1)
[9/10] AppID 2694490... +1 (total: 1)
[10/10] AppID 1086940... +1 (total: 1)
💾 Checkpoint: 10 registros | 2s

🔄 RONDA 2 - 10 activos
💾 Checkpoint: 10 registros | 2s

✅ Completado: 10 registros en 0.0 min
   Steam IDs únicos: 10


> Con la ejecución pasada se realizaron pruebas para ir obteniendo id's solo con los juegos que si tenian reviews.
> - rpg_steamids_prueba (ejecución anterior)
> - rpg_steamids_prueba_252490
> - rpg_steamids_appid_1623730
> - rpg_steamids_all_final

In [ ]:
# Obtenemos los id de los juegos que si tienen reviews
rpg_appids = df_steamids_prueba['game_appid'].unique().tolist()

> - df_steamids_252490: Se obtuvieron 704K reviews, en 7K rondas, obtuvimos 4.5K jugadores

In [ ]:
# Aproximadamente 1hr de ejcución
df_steamids_252490 = get_reviews_for_multiple_games_progressive(
    appids=[252490],
    checkpoint_file="data/rpg_steamids_all_prueba_252490.parquet",
    batch_size=100,
    max_reviews_per_game=None,
    checkpoint_every_round=1,
    rate_limit_seconds=1.0,
    language=None # Todos los lenguajes
)

📂 Datos cargados: 1,500 registros
📂 Reanudando ronda 16

🎮 1 juegos | Batch: 100

🔄 RONDA 16 - 1 activos
💾 Checkpoint: 1,500 registros | 0s

✅ Completado: 1,500 registros en 0.0 min
   Steam IDs únicos: 1,500


> - rpg_steamids_appid_1623730: Se obtuvieron 100K reviews, en 1K rondas, obtuvimos 273 jugadores

In [ ]:
# Aproximandamente 20 minutos de ejcución
df_steamids_1623730 = get_reviews_for_multiple_games_progressive(
    appids=[1623730],
    checkpoint_file="data/rpg_steamids_all_prueba_1623730.parquet",
    batch_size=100,
    max_reviews_per_game=None,
    checkpoint_every_round=1,
    rate_limit_seconds=1.0,
    language=None
)


🎮 1 juegos | Batch: 100

🔄 RONDA 1 - 1 activos
[1/1] AppID 1623730... +100 (total: 100)
💾 Checkpoint: 100 registros | 0s

🔄 RONDA 2 - 1 activos
💾 Checkpoint: 100 registros | 0s

✅ Completado: 100 registros en 0.0 min
   Steam IDs únicos: 100


> - rpg_steamids_all_final: Se obtuvieron 190K reviews, se ejecutaron 43 rondas, obtuvimos 3.4K jugadores.

In [ ]:
df_steamids_final = get_reviews_for_multiple_games_progressive(
    appids=rpg_appids,
    checkpoint_file="data/rpg_steamids_all_final.parquet",
    batch_size=1,
    max_reviews_per_game=None,
    checkpoint_every_round=1,
    rate_limit_seconds=1.0
)


🎮 10 juegos | Batch: 1

🔄 RONDA 1 - 10 activos
[1/10] AppID 1623730... +1 (total: 1)
[2/10] AppID 1063730... +1 (total: 1)
[3/10] AppID 2358720... +1 (total: 1)
[4/10] AppID 1599340... +1 (total: 1)
[5/10] AppID 2246340... +1 (total: 1)
[6/10] AppID 230410... +1 (total: 1)
[7/10] AppID 1245620... +1 (total: 1)
[8/10] AppID 105600... +1 (total: 1)
[9/10] AppID 2694490... +1 (total: 1)
[10/10] AppID 1086940... +1 (total: 1)
💾 Checkpoint: 10 registros | 2s

🔄 RONDA 2 - 10 activos
💾 Checkpoint: 10 registros | 2s

✅ Completado: 10 registros en 0.0 min
   Steam IDs únicos: 10


Unimos todos los id, para hacer la busqueda de sus amigos

In [ ]:
data_frame_jugadores = pd.concat([df_steamids_prueba, df_steamids_252490, df_steamids_1623730, df_steamids_final])

In [ ]:
data_frame_jugadores['game_appid'] = data_frame_jugadores['game_appid'].astype(str)
data_frame_jugadores['author_steamid'] = data_frame_jugadores['author_steamid'].astype(str)

In [ ]:
data_frame_jugadores.to_parquet(r'./data/data_frame_jugadores.parquet', index=False)

# Amigos de jugadores RPG

Obtenemos mas id's de jugadores mediante una api que nos da la lista de amigos de los id's de jugadores que ya obtuvimos con la api de reviews, esto para aumentar nuetro segmento.

In [ ]:
# Lista de Steam IDs a procesar (ejemplo - reemplaza con tus IDs)
steam_ids_to_process = data_frame_jugadores['author_steamid'].unique().tolist()

In [ ]:
df_amigos = process_steam_ids_with_checkpoint(
    steam_ids_list=steam_ids_to_process,
    api_key=STEAM_API_KEY,
    checkpoint_file='data/amigos_steam.parquet',
    checkpoint_interval=10,  # Guardar cada 10 Steam IDs procesados
    delay=1.5  # Esperar 1.5 segundos entre requests
)

🆕 Iniciando proceso desde cero
📊 Total a procesar: 10 Steam IDs
[1/10] Procesando Steam ID: 76561198316252062... ✓ 18 amigos encontrados
[2/10] Procesando Steam ID: 76561198102361106... ❌ Error 401: API key inválida - verifica STEAM_API_KEY
⚠️  Error (total errores: 1), continuando...
[3/10] Procesando Steam ID: 76561199446716747... ✓ 6 amigos encontrados
[4/10] Procesando Steam ID: 76561198055301153... ✓ 34 amigos encontrados
[5/10] Procesando Steam ID: 76561198052632518... ✓ 20 amigos encontrados
[6/10] Procesando Steam ID: 76561199241093426... ✓ 8 amigos encontrados
[7/10] Procesando Steam ID: 76561199161007226... ✓ 9 amigos encontrados
[8/10] Procesando Steam ID: 76561199089545468... ✓ 11 amigos encontrados
[9/10] Procesando Steam ID: 76561198188719623... ✓ 10 amigos encontrados
[10/10] Procesando Steam ID: 76561199056975880... ✓ 23 amigos encontrados
💾 Checkpoint guardado (140 registros totales)

✅ Proceso completado. Total de registros: 140
💾 Guardado en: data/amigos_steam.parque

# Tiempo Jugado en Juegos

Obtenemos 2 metricas importantes de la api de Steam, las horas totales Jugadas en las ultimas 2 semanas y las horas totales jugadas historicamente.

Obtenemos una lista unica de los id de steam para obtener las horas jugadas en cada juego

In [ ]:
jugadores_unicos = set(df_amigos['steamid'].unique().tolist()).union(set(data_frame_jugadores['author_steamid'].unique().tolist()))

In [ ]:
jugadores_unicos = list(jugadores_unicos)[:10]

In [ ]:
df_jugadores_para_busqueda = pd.DataFrame({'author_steamid': jugadores_unicos})

> Con los id de las reseñas convinado con los id de los amigos de las reseñas obtuvimos mas de 243K jugadores, esta parte demora mucho en ejecutarse, solo se tomara un segmento.

In [ ]:
# Mas de 1 día de ejecución
df_juegos_jugadores_steam = obtener_juegos_todos_jugadores_parquet(
    df_jugadores=df_jugadores_para_busqueda,
    api_key=STEAM_API_KEY,
    checkpoint_file='data/checkpoint_juegos_jugadores_steam.parquet',
    max_jugadores=None,  # TODOS los jugadores
    checkpoint_cada=10,  # Guarda cada 10 jugadores
    debug=True
)

🎮 Jugadores: 10 | Ya procesados: 0 | Pendientes: 10

✅ [1/10] 76561199057367219: 225 juegos
Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:
https://steamcommunity.com/my/edit/settings
🔒 [2/10] 76561198312640962: Perfil privado
Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:
https://steamcommunity.com/my/edit/settings
🔒 [3/10] 76561199468805298: Perfil privado
✅ [4/10] 76561199817315135: 14 juegos
Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:
https://steamcommunity.com/my/edit/settings
🔒 [5/10] 76561198390844104: Perfil privado
Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:
https://steamcommunity.com/my/edit/settings
🔒 [6/10] 76561199754511636: Perfil privado
Perfil privado o sin juegos. El usuario debe hacer público 'Game details' en:
https://steamcommunity.com/my/edit/settings
🔒 [7/10] 76561199416703078: Perfil privado
Perfil privado o sin juegos. El usuario debe 

# EDA y Visualizaciónes

- Limpieza de nulos
- Detección de valores atipicos
- Analisis de Duplicados

# Ingenieria de Variables

- Obtener estadisticas generales del comportamiento en Steam y estadisticas de Juegos RPG a nivel global y por juego
- Obtención de insights y visualizaciónes de transformaciones (cosas interesantes)